# Capstone Data Collection
**Project:** Predicting Long-Run Home Price Appreciation Across U.S. Housing Markets  
**Notebook:** 01: Data Collection  

This notebook collects all source data for the project:
1. **FHFA HPI**: Quarterly home price index for 100 largest MSAs (1991–present)
2. **Census ACS**: Annual MSA-level demographic and economic indicators (2011–2024)
3. **FRED**: National 30-year fixed mortgage rate (2010–present)

### Census Variables Collected (ACS 1-Year Estimates)
| Table | Variables | Derived Field |
|-------|-----------|---------------|
| B01003 | `_001E` | Total population |
| B23025 | `_002E`, `_005E` | Unemployment rate |
| B17001 | `_001E`, `_002E` | Poverty rate |
| B19013 | `_001E` | Median household income |
| B15003 | `_001E`, `_022E`–`_025E` | % with bachelor's degree or higher |
| B25002 | `_001E`, `_003E` | Vacancy rate |
| B25003 | `_001E`, `_002E` | Homeownership rate |

**Note:** 2020 ACS 1-year estimates were not published due to COVID-related data collection
disruptions. Year 2020 is intentionally excluded from all Census pulls.

All raw data is saved to `data/raw/` for use in the next notebook.

## Setup

In [10]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

# Load API keys from .env file
load_dotenv()
CENSUS_API_KEY = os.getenv('CENSUS_API_KEY')
FRED_API_KEY   = os.getenv('FRED_API_KEY')

if not CENSUS_API_KEY or not FRED_API_KEY:
    raise ValueError('API keys not found. Make sure .env file exists with CENSUS_API_KEY and FRED_API_KEY.')

# Create directory structure
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print('Setup complete. API keys loaded.')

Setup complete. API keys loaded.


---
## 1. FHFA House Price Index: 100 Largest MSAs

Downloaded manually from:  
https://www.fhfa.gov/data/hpi/datasets?tab=quarterly-data  
-> Purchase-Only Indexes -> 100 Largest Metropolitan Statistical Areas (Seasonally Adjusted)

In [11]:
hpi = pd.read_excel('data/raw/hpi_po_metro.xlsx')

# Standardize CBSA code as zero-padded 5-digit string for merging
hpi['cbsa'] = hpi['cbsa'].astype(str).str.zfill(5)

print(f'HPI shape: {hpi.shape}')
print(f'Years:     {hpi["yr"].min()} – {hpi["yr"].max()}')
print(f'MSAs:      {hpi["metro_name"].nunique()}')
print(f'Columns:   {hpi.columns.tolist()}')
print()
hpi.head()

HPI shape: (14000, 6)
Years:     1991 – 2025
MSAs:      100
Columns:   ['cbsa', 'metro_name', 'yr', 'qtr', 'index_nsa', 'index_sa']



,cbsa,metro_name,yr,qtr,index_nsa,index_sa
0,10580,"Albany-Schenectady-Troy, NY",1991,1,100.00,100.00
1,10580,"Albany-Schenectady-Troy, NY",1991,2,99.85,100.31
2,10580,"Albany-Schenectady-Troy, NY",1991,3,100.20,100.42
3,10580,"Albany-Schenectady-Troy, NY",1991,4,99.83,98.80
4,10580,"Albany-Schenectady-Troy, NY",1992,1,102.80,102.86


---
## 2. Census ACS: MSA-Level Indicators (2011–2024)

All seven indicator groups are pulled in a single API call per year,
minimizing requests.

In [12]:
# -------------------------------------------------------------------------
# Census variable definitions
# Each entry: (raw column name from API, human-readable label)
# -------------------------------------------------------------------------
ACS_VARS = {
    # Population
    'B01003_001E': 'population',

    # Labor force & unemployment (B23025)
    'B23025_002E': 'labor_force',
    'B23025_005E': 'unemployed',

    # Poverty (B17001)
    'B17001_001E': 'poverty_universe',   # total pop for whom poverty status determined
    'B17001_002E': 'poverty_count',      # count below poverty line

    # Median household income (B19013)
    'B19013_001E': 'median_income',

    # Educational attainment 25+ (B15003)
    'B15003_001E': 'edu_universe',       # total pop 25 and over
    'B15003_022E': 'edu_bachelors',      # bachelor's degree
    'B15003_023E': 'edu_masters',        # master's degree
    'B15003_024E': 'edu_professional',   # professional school degree
    'B15003_025E': 'edu_doctorate',      # doctorate degree

    # Housing occupancy / vacancy (B25002)
    'B25002_001E': 'housing_units_total',
    'B25002_003E': 'housing_units_vacant',

    # Housing tenure / homeownership (B25003)
    'B25003_001E': 'tenure_total',       # total occupied housing units
    'B25003_002E': 'tenure_owner',       # owner-occupied units
}

get_vars = 'NAME,' + ','.join(ACS_VARS.keys())
print(f'Pulling {len(ACS_VARS)} ACS variables per MSA per year.')
print(f'Variable string: {get_vars[:120]}...')

Pulling 15 ACS variables per MSA per year.
Variable string: NAME,B01003_001E,B23025_002E,B23025_005E,B17001_001E,B17001_002E,B19013_001E,B15003_001E,B15003_022E,B15003_023E,B15003_...


In [13]:
def get_acs_year(year, api_key, variables):
    """
    Pull Census ACS 1-year estimates for all MSAs for a given year.

    Parameters
    ----------
    year      : int   - survey year (2011–2024, excluding 2020)
    api_key   : str   - Census API key
    variables : str   - comma-separated variable codes (including NAME)

    Returns
    -------
    pd.DataFrame or None if request fails
    """
    url = f'https://api.census.gov/data/{year}/acs/acs1'
    params = {
        'get': variables,
        'for': 'metropolitan statistical area/micropolitan statistical area:*',
        'key': api_key
    }

    try:
        r = requests.get(url, params=params)
        if r.status_code != 200:
            print(f'  [{year}] HTTP {r.status_code} ...skipping')
            return None
        data = r.json()
        df = pd.DataFrame(data[1:], columns=data[0])
        df['year'] = year
        return df
    except Exception as e:
        print(f'  [{year}] Error: {e} ...skipping')
        return None


# 2020 excluded: no standard ACS 1-year release published that year due to COVID
years = [y for y in range(2011, 2025) if y != 2020]

print(f'Pulling ACS data for {len(years)} years ({years[0]}–{years[-1]}, 2020 excluded)...\n')

frames = []
for year in years:
    print(f'  {year}...', end=' ')
    df = get_acs_year(year, CENSUS_API_KEY, get_vars)
    if df is not None:
        frames.append(df)
        print(f'{len(df):,} MSAs')

acs_raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal raw rows: {len(acs_raw):,}')

Pulling ACS data for 13 years (2011–2024, 2020 excluded)...

  2011... 529 MSAs
  2012... 530 MSAs
  2013... 515 MSAs
  2014... 515 MSAs
  2015... 516 MSAs
  2016... 518 MSAs
  2017... 518 MSAs
  2018... 519 MSAs
  2019... 518 MSAs
  2021... 513 MSAs
  2022... 516 MSAs
  2023... 530 MSAs
  2024... 535 MSAs

Total raw rows: 6,772


In [14]:
acs_raw.sample(10)

,NAME,B01003_001E,B23025_002E,B23025_005E,B17001_001E,B17001_002E,B19013_001E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,B25002_001E,B25002_003E,B25003_001E,B25003_002E,metropolitan statistical area/micropolitan statistical area,year
600,"Cape Girardeau-Jackson, MO-IL Metro Area",96410,47534,3687,92459,16386,42686,63131,10510,3518,760,801,42085,4473,37612,25888,16020,2012
5602,"Sandusky, OH Micro Area",74501,39677,1389,73089,7890,69296,53652,9381,4147,912,502,38514,5060,33454,24544,41780,2022
3036,"Dunn, NC Micro Area",130881,61258,3721,127123,20298,51637,84127,11089,4376,834,436,50611,6651,43960,27637,20380,2016
4667,"Winston-Salem, NC Metro Area",676008,325516,14867,662172,105816,52322,471015,81489,32805,7880,5338,303972,33699,270273,183020,49180,2019
615,"Chico, CA Metro Area",221539,101652,15249,216391,47693,40960,142262,22877,7207,2642,1525,96000,10612,85388,50752,17020,2012
4601,"Sioux City, IA-NE-SD Metro Area",144670,76922,2306,141339,16314,60132,94806,14578,4400,1664,790,60583,4491,56092,37225,43580,2019
3418,"Medford, OR Metro Area",217479,102595,5891,215025,30298,51409,154281,27944,10587,3645,1587,95394,5693,89701,56944,32780,2017
6160,"Somerset, KY Micro Area",66191,27450,1164,64972,10174,55667,46664,3377,3854,750,515,31793,5590,26203,18997,43700,2023
1257,"Torrington, CT Micro Area",186924,107561,7863,184394,12330,70981,136058,27007,14618,3173,1440,87262,12233,75029,56798,45860,2013
3493,"Ponce, PR Metro Area",300059,96472,20974,294772,155484,16109,205946,37440,9497,809,2200,135907,32522,103385,71234,38660,2017


In [15]:
# Clean raw columns and compute all rates

geo_col = 'metropolitan statistical area/micropolitan statistical area'

# Rename geography and name columns
acs = acs_raw.rename(columns={geo_col: 'cbsa', 'NAME': 'metro_name_acs'})

# Rename raw Census codes to readable labels
acs = acs.rename(columns=ACS_VARS)

# Zero-pad CBSA to 5 char for consistent merging with HPI
acs['cbsa'] = acs['cbsa'].astype(str).str.zfill(5)

# Convert all value columns to numeric
# (Census API returns everything as strings; -666666666 is the Census null sentinel)
value_cols = list(ACS_VARS.values())
for col in value_cols:
    acs[col] = pd.to_numeric(acs[col], errors='coerce')
    acs[col] = acs[col].where(acs[col] != -666666666, other=pd.NA)

# --- Calculate rates ---

# Unemployment rate: unemployed / labor force × 100
acs['unemployment_rate'] = (acs['unemployed'] / acs['labor_force']) * 100

# Poverty rate: pop below poverty / poverty universe × 100
acs['poverty_rate'] = (acs['poverty_count'] / acs['poverty_universe']) * 100

# Pct with bachelor's degree or higher: sum of 4 degree levels / edu universe × 100
acs['pct_bachelors_plus'] = (
    (acs['edu_bachelors'] + acs['edu_masters'] + acs['edu_professional'] + acs['edu_doctorate'])
    / acs['edu_universe']
) * 100

# Vacancy rate: vacant units / total housing units × 100
acs['vacancy_rate'] = (acs['housing_units_vacant'] / acs['housing_units_total']) * 100

# Homeownership rate: owner-occupied / total occupied × 100
acs['homeownership_rate'] = (acs['tenure_owner'] / acs['tenure_total']) * 100

# -------------------------------------------------------------------------
# Keep final columns only
# -------------------------------------------------------------------------
final_cols = [
    'cbsa', 
    'metro_name_acs', 
    'year',
    'population',
    'unemployment_rate',
    'poverty_rate',
    'median_income',
    'pct_bachelors_plus',
    'vacancy_rate',
    'homeownership_rate'
]

acs = acs[final_cols]

# Drop rows missing the three most critical fields
acs = acs.dropna(subset=['population', 'unemployment_rate', 'median_income'])

print(f'ACS shape after cleaning: {acs.shape}')
print(f'Years: {acs["year"].min()} – {acs["year"].max()}')
print(f'Unique MSAs: {acs["cbsa"].nunique()}')
print()
print('Missing value summary:')
print(acs[final_cols[3:]].isna().sum())
print()
acs.sample(10)

ACS shape after cleaning: (6772, 10)
Years: 2011 – 2024
Unique MSAs: 594

Missing value summary:
population             0
unemployment_rate      0
poverty_rate           1
median_income          0
pct_bachelors_plus    20
vacancy_rate           0
homeownership_rate     0
dtype: int64



,cbsa,metro_name_acs,year,population,unemployment_rate,poverty_rate,median_income,pct_bachelors_plus,vacancy_rate,homeownership_rate
2927,48580,"Whitewater-Elkhorn, WI Micro Area",2016,102959,4.326488,11.164724,58302,29.234738,22.862482,66.877295
4233,16220,"Casper, WY Metro Area",2019,79858,2.590808,8.570073,65034,20.672439,11.547727,72.436751
3765,19820,"Detroit-Warren-Dearborn, MI Metro Area",2018,4326442,5.625446,14.314598,60513,31.754221,10.569013,69.131799
1834,30340,"Lewiston-Auburn, ME Metro Area",2014,107440,5.885017,15.563016,48007,22.938527,9.920231,65.488965
6512,30700,"Lincoln, NE Metro Area",2024,351919,3.286709,12.355634,74935,43.915515,4.162927,60.665609
5345,22660,"Fort Collins, CO Metro Area",2022,366778,3.990947,11.579643,88403,52.935249,6.159550,63.959039
18,11300,"Anderson, IN Metro Area",2011,131235,13.475473,19.551344,42129,18.074389,15.626273,70.490616
1292,48700,"Williamsport, PA Metro Area",2013,116754,9.405881,13.513440,47373,18.957992,14.211300,72.235938
4324,23580,"Gainesville, GA Metro Area",2019,204441,3.844587,14.534103,67467,25.381541,13.840640,73.598476
4567,41400,"Salem, OH Micro Area",2019,101883,5.618605,12.440118,53542,15.541748,13.429523,75.522682


In [16]:
# Sanity check: inspect one MSA across several years
sample = acs[acs['metro_name_acs'].str.contains('Atlanta', na=False)].sort_values('year')
print('Atlanta MSA (all years):')
print(sample[['metro_name_acs', 'year', 'unemployment_rate', 'poverty_rate',
              'median_income', 'pct_bachelors_plus', 'vacancy_rate',
              'homeownership_rate']].to_string(index=False))

Atlanta MSA (all years):
                                 metro_name_acs  year  unemployment_rate  poverty_rate  median_income  pct_bachelors_plus  vacancy_rate  homeownership_rate
  Atlanta-Sandy Springs-Marietta, GA Metro Area  2011          12.659529     16.799096          52639           34.540227     12.617605           64.336341
  Atlanta-Sandy Springs-Marietta, GA Metro Area  2012          11.073592     16.589347          54628           35.304472     11.565102           63.812017
   Atlanta-Sandy Springs-Roswell, GA Metro Area  2013           9.831952     15.945711          55733           35.156516     10.832188           63.317741
   Atlanta-Sandy Springs-Roswell, GA Metro Area  2014           7.855673     15.507950          56166           35.842263     10.518056           62.411864
   Atlanta-Sandy Springs-Roswell, GA Metro Area  2015           6.718311     13.902057          60219           37.035996      9.434511           61.552863
   Atlanta-Sandy Springs-Roswell, GA Me

---
## 3. FRED: National 30-Year Fixed Mortgage Rate

Series: `MORTGAGE30US` (Freddie Mac Primary Mortgage Market Survey)  
Frequency: Weekly (averaged to quarterly to align with HPI cadence)

This is a national predictor. The same value applies to all MSAs in a given quarter,
capturing macro-level financing conditions that affect home affordability universally.

In [17]:
def get_fred_series(series_id, api_key, start='2010-01-01', frequency='q', aggregation='avg'):
    """
    Pull a FRED series at a specified frequency.

    Parameters
    ----------
    series_id   : str - FRED series identifier (e.g. 'MORTGAGE30US')
    api_key     : str - FRED API key
    start       : str - observation start date (YYYY-MM-DD)
    frequency   : str - 'q' quarterly, 'a' annual, 'm' monthly
    aggregation : str - 'avg', 'sum', 'eop' (end of period)

    Returns
    -------
    pd.DataFrame with columns [yr, qtr, <series_id_lower>]
    """
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id': series_id,
        'api_key': api_key,
        'file_type': 'json',
        'frequency': frequency,
        'aggregation_method': aggregation,
        'observation_start': start
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    obs = r.json()['observations']
    df = pd.DataFrame(obs)[['date', 'value']]
    df['date'] = pd.to_datetime(df['date'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['yr'] = df['date'].dt.year
    df['qtr'] = df['date'].dt.quarter
    df = df.rename(columns={'value': series_id.lower()})
    return df[['yr', 'qtr', series_id.lower()]].dropna()


mortgage = get_fred_series('MORTGAGE30US', FRED_API_KEY)

print(f'Mortgage rate rows: {len(mortgage)}')
print(f'Range: {mortgage["yr"].min()} Q{int(mortgage.loc[mortgage["yr"].idxmin(), "qtr"])} '
      f'– {mortgage["yr"].max()} Q{int(mortgage.loc[mortgage["yr"].idxmax(), "qtr"])}')
print()
mortgage.tail(8)

Mortgage rate rows: 66
Range: 2010 Q1 – 2026 Q1



,yr,qtr,mortgage30us
58,2024,3,6.51
59,2024,4,6.63
60,2025,1,6.83
61,2025,2,6.79
62,2025,3,6.57
63,2025,4,6.23
64,2026,1,6.11
65,2026,2,6.40


---
## 4. Save Raw Data

In [18]:
hpi.to_csv('data/raw/hpi_po_metro.csv', index=False)
acs.to_csv('data/raw/acs_indicators.csv', index=False)
mortgage.to_csv('data/raw/fred_mortgage30us.csv', index=False)

print('Saved to data/raw/:')
print('  hpi_po_metro.csv')
print('  acs_indicators.csv')
print('  fred_mortgage30us.csv')
print()
print('Summary')
print('-' * 55)
print(f'  HPI       {hpi.shape[0]:>6,} rows | {hpi["metro_name"].nunique()} MSAs | {hpi["yr"].min()}–{hpi["yr"].max()}')
print(f'  ACS       {acs.shape[0]:>6,} rows | {acs["cbsa"].nunique()} MSAs | {acs["year"].min()}–{acs["year"].max()} (no 2020)')
print(f'  Mortgage  {mortgage.shape[0]:>6,} rows | national quarterly series')
print()
print('ACS columns saved:')
for col in acs.columns:
    print(f'  {col}')

Saved to data/raw/:
  hpi_po_metro.csv
  acs_indicators.csv
  fred_mortgage30us.csv

Summary
-------------------------------------------------------
  HPI       14,000 rows | 100 MSAs | 1991–2025
  ACS        6,772 rows | 594 MSAs | 2011–2024 (no 2020)
  Mortgage      66 rows | national quarterly series

ACS columns saved:
  cbsa
  metro_name_acs
  year
  population
  unemployment_rate
  poverty_rate
  median_income
  pct_bachelors_plus
  vacancy_rate
  homeownership_rate
